# Build the preprocessing pipeline

One row is one telco customer; `Churn` says whether they left in the last month. You fix the `TotalCharges` column, then put every preparation step (impute, scale, encode) inside one `ColumnTransformer` that learns only from the training rows. The README shows the code for every step.

## Step 1 · Load the table and look at `TotalCharges`

In [1]:
import pandas as pd

df = pd.read_csv("data/telco_churn.csv")
df["TotalCharges"].dtype

<StringDtype(na_value=nan)>

## Step 2 · Turn `TotalCharges` into numbers

The 11 blanks become `NaN`, which the pipeline fills in step 5.

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].isna().sum()

np.int64(11)

## Step 3 · Separate the target, drop the identifier, list the two kinds of column

In [3]:
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
print(len(num_cols), "numeric:", num_cols)
print(len(cat_cols), "text:", cat_cols)

4 numeric: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
15 text: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## Step 4 · Split before anything is fitted

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
print(len(X_train), "training rows,", X_train["TotalCharges"].isna().sum(), "with TotalCharges missing")
print(len(X_test), "test rows")

5634 training rows, 8 with TotalCharges missing
1409 test rows


## Step 5 · Build the `ColumnTransformer` and fit it on the training rows

Numbers get the median and a scale; text gets one-hot encoding.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
pre.set_output(transform="pandas")
Xt = pre.fit_transform(X_train)
Xt.shape, Xt.isna().sum().sum()

((5634, 45), np.int64(0))

## Step 6 · Put a model at the end and score it against a dummy

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

model = make_pipeline(pre, LogisticRegression(max_iter=1000)).fit(X_train, y_train)
print(model.score(X_test, y_test))
print(DummyClassifier().fit(X_train, y_train).score(X_test, y_test))

0.8055358410220014
0.7345635202271115


## The median, and why 0 is the better fill

The imputer learned its median from the training rows only. The 11 blanks are customers with tenure 0, who have not been billed yet, so 0 is what the blank means; filling with 0 gives the same accuracy here.

In [7]:
imputer = model[0].named_transformers_["num"][0]
median_train = imputer.statistics_[num_cols.index("TotalCharges")]
print("median learned from the training rows:", round(median_train, 2))
print("median over all rows (not allowed):   ", round(df["TotalCharges"].median(), 2))
print("tenure of the blank rows:", df.loc[df["TotalCharges"].isna(), "tenure"].unique())

from sklearn.base import clone

Xz_train, Xz_test = X_train.fillna({"TotalCharges": 0}), X_test.fillna({"TotalCharges": 0})
zero = make_pipeline(clone(pre), LogisticRegression(max_iter=1000)).fit(Xz_train, y_train)
print("accuracy with the blanks filled with 0:", round(zero.score(Xz_test, y_test), 3))

median learned from the training rows: 1398.12
median over all rows (not allowed):    1397.48
tenure of the blank rows: [0]
accuracy with the blanks filled with 0: 0.806


## Check yourself

In [8]:
check = {
    "Step 1 · TotalCharges dtype as read": pd.read_csv("data/telco_churn.csv")["TotalCharges"].dtype,
    "Step 2 · blanks turned into NaN": df["TotalCharges"].isna().sum(),
    "Step 3 · numeric columns": len(num_cols),
    "Step 3 · numeric column names": ", ".join(num_cols),
    "Step 3 · text columns": len(cat_cols),
    "Step 4 · training rows": len(X_train),
    "Step 4 · training rows with TotalCharges missing": X_train["TotalCharges"].isna().sum(),
    "Step 4 · test rows": len(X_test),
    "Step 5 · shape after the transformer": Xt.shape,
    "Step 5 · missing values after": Xt.isna().sum().sum(),
    "Step 6 · model accuracy": round(model.score(X_test, y_test), 3),
    "Step 6 · dummy accuracy": round(DummyClassifier().fit(X_train, y_train).score(X_test, y_test), 3),
    "Median TotalCharges, training rows": round(median_train, 2),
    "Median TotalCharges, all rows": round(df["TotalCharges"].median(), 2),
    "Accuracy with the blanks filled with 0": round(zero.score(Xz_test, y_test), 3),
}
for label, value in check.items():
    print(f"{label:<50} {value}")

Step 1 · TotalCharges dtype as read                str
Step 2 · blanks turned into NaN                    11
Step 3 · numeric columns                           4
Step 3 · numeric column names                      SeniorCitizen, tenure, MonthlyCharges, TotalCharges
Step 3 · text columns                              15
Step 4 · training rows                             5634
Step 4 · training rows with TotalCharges missing   8
Step 4 · test rows                                 1409
Step 5 · shape after the transformer               (5634, 45)
Step 5 · missing values after                      0
Step 6 · model accuracy                            0.806
Step 6 · dummy accuracy                            0.735
Median TotalCharges, training rows                 1398.12
Median TotalCharges, all rows                      1397.48
Accuracy with the blanks filled with 0             0.806
